## wifi signal for indoor localisation ##

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import preprocessing

In [ ]:
df = pd.read_csv("Wifi_train_dataset.csv")

# finding and dropping NaN values
print(f"Initial no. of rows: {len(df)}")
print(f"NaN values:\n{df.isna().sum()}")
df.dropna()
print(f"Final no. of rows: {len(df)}")
df.head(5)

X = df["x"]
Y = df["y"]

# make RSSI, MAC into integer type
df["rssi_list"] = df["rssi"].apply(lambda x: [int(rssi) for rssi in x.split(",")])
df["mac_list"] = df["mac_addrs_idx"].apply(lambda x: [int(mac) for mac in x.split(",")])

# all beacons/routers (unique MAC addresses)
mac_unique = set()
for mac in df["mac_list"]:
    mac_unique.update(mac)
    
mac_unique = sorted(mac_unique)
print(f"{len(mac_unique)} unique MAC values: {mac_unique}")

# create a dict which maps each MAC idx to a column no.
mac_to_col = {mac: i for i, mac in enumerate(mac_unique)}
num_features = len(mac_to_col)

# making feature matrix for multiple linear regression
# preallocate matrix
X_train = np.full((len(df), num_features), -101, dtype=np.float64)        # default RSSI is -101 for undetected


for i, (rssi_list, mac_list) in enumerate(zip(df['rssi_list'], df['mac_list'])):
    for rssi, mac in zip(rssi_list, mac_list):
        # X_train is RSSI value
        X_train[i, mac_to_col[mac]] = rssi        # assigns corresponding RSSI to the right row and col (mac column)
        
# y_train is [X;Y]
y_train = df[['x', 'y']].values

print(X_train.shape)
print(y_train.shape)

Initial no. of rows: 2880
NaN values:
x                0
y                0
rssi             0
mac_addrs_idx    0
dtype: int64
Final no. of rows: 2880
562 unique MAC values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187

In [54]:
# clean again just in case
nan_mask = np.isnan(y_train).any(axis=1)
print(f"Rows with NaN for (x, y): {nan_mask.sum()}")
# so no need to clean

# split data to train/validation split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# make model and train!
model = LinearRegression()

model.fit(X_train,y_train)

Rows with NaN for (x, y): 0
Training set: 1843 samples
Validation set: 461 samples


LinearRegression()

In [ ]:
# validate
y_val = model.predict(X_val)

# Calculate MSE for each coordinate
mse_x = root_mean_squared_error(y_val[:, 0], y_val[:, 0])
mse_y = root_mean_squared_error(y_val[:, 1], y_val[:, 1])
mse_total = root_mean_squared_error(y_val, y_val)